In [1]:
from pathlib import Path

# base_path = Path("deepsort_object_tracking")
# input_video_path = base_path / "input_video"

# input_video_path.mkdir(parents=True, exist_ok=True)
# from pathlib import Path

base_path = Path("deepsort_object_tracking")
input_video_path = base_path / "output_video"

input_video_path.mkdir(parents=True, exist_ok=True)

In [2]:
%%writefile deepsort_object_tracking/detection.py
import numpy as np


class Detection:
    """
    Represents one detection in one frame.

    bbox:       [x1, y1, x2, y2]
    confidence: float
    feature:    128-dim appearance embedding (numpy array)
    """

    def __init__(self, bbox, confidence, feature):
        self.bbox       = np.asarray(bbox, dtype=np.float32)
        self.confidence = float(confidence)
        self.feature    = np.asarray(feature, dtype=np.float32)

    def to_tlwh(self):
        """Convert [x1,y1,x2,y2] to [top,left,width,height]"""
        ret = self.bbox.copy()
        ret[2] -= ret[0]   # w = x2 - x1
        ret[3] -= ret[1]   # h = y2 - y1
        return ret

    def to_xyah(self):
        """
        Convert to [center_x, center_y, aspect_ratio, height]
        This is the Kalman measurement format
        """
        ret = self.to_tlwh()
        ret[:2] += ret[2:] / 2    # center
        ret[2]  /= ret[3]         # aspect ratio = w/h
        return ret

Writing deepsort_object_tracking/detection.py


In [3]:
%%writefile deepsort_object_tracking/kalman_filter.py
import numpy as np
import scipy.linalg


# Chi-squared distribution 95th percentile
# 4 degrees of freedom (measurement dim)
# Used for Mahalanobis gating threshold
chi2inv95 = {
    1: 3.8415,
    2: 5.9915,
    3: 7.8147,
    4: 9.4877,   # ← we use this (dim_z=4)
    5: 11.0705,
    6: 12.5916,
    7: 14.0671,
    8: 15.5073,
    9: 16.9190
}


class KalmanFilter:
    """
    Kalman filter for bounding box tracking.

    State:       [u, v, γ, h, u̇, v̇, γ̇, ḣ]  (8-dim)
    Measurement: [u, v, γ, h]               (4-dim)
    """

    def __init__(self):
        ndim = 4   # measurement dims
        dt   = 1   # time step (one frame)

        # ── State Transition Matrix F (8×8) ──────────────
        # x_new = F @ x_old
        self._motion_mat = np.eye(2 * ndim, 2 * ndim)
        for i in range(ndim):
            self._motion_mat[i, ndim + i] = dt
        # Result:
        # [1,0,0,0, 1,0,0,0]   u  = u  + u_dot
        # [0,1,0,0, 0,1,0,0]   v  = v  + v_dot
        # [0,0,1,0, 0,0,1,0]   γ  = γ  + γ_dot
        # [0,0,0,1, 0,0,0,1]   h  = h  + h_dot
        # [0,0,0,0, 1,0,0,0]   u_dot constant
        # [0,0,0,0, 0,1,0,0]   v_dot constant
        # [0,0,0,0, 0,0,1,0]   γ_dot constant
        # [0,0,0,0, 0,0,0,1]   h_dot constant

        # ── Measurement Matrix H (4×8) ────────────────────
        # z = H @ x
        self._update_mat = np.eye(ndim, 2 * ndim)
        # Result:
        # [1,0,0,0, 0,0,0,0]   observe u
        # [0,1,0,0, 0,0,0,0]   observe v
        # [0,0,1,0, 0,0,0,0]   observe γ
        # [0,0,0,1, 0,0,0,0]   observe h

        # ── Noise Scale Factors ───────────────────────────
        # These scale the identity matrices for Q and R
        self._std_weight_position = 1. / 20    # position noise
        self._std_weight_velocity = 1. / 160   # velocity noise

    def initiate(self, measurement):
        """
        Create initial state from first measurement.

        measurement: [u, v, γ, h]
        Returns: (mean, covariance) — initial state estimate
        """
        mean_pos = measurement                    # [u, v, γ, h]
        mean_vel = np.zeros_like(mean_pos)        # [0, 0, 0, 0]
        mean     = np.r_[mean_pos, mean_vel]      # [u,v,γ,h,0,0,0,0]

        # Initial uncertainty — high for velocities
        std = [
            2 * self._std_weight_position * measurement[3],   # u
            2 * self._std_weight_position * measurement[3],   # v
            1e-2,                                              # γ
            2 * self._std_weight_position * measurement[3],   # h
            10 * self._std_weight_velocity * measurement[3],  # u_dot
            10 * self._std_weight_velocity * measurement[3],  # v_dot
            1e-5,                                              # γ_dot
            10 * self._std_weight_velocity * measurement[3],  # h_dot
        ]

        covariance = np.diag(np.square(std))   # P = diag(std²)
        return mean, covariance

    def predict(self, mean, covariance):
        """
        Kalman PREDICT step.

        x̂ = F @ x
        P̂ = F @ P @ Fᵀ + Q

        Returns: (predicted_mean, predicted_covariance)
        """
        # Process noise Q — how much can state change unexpectedly
        std_pos = [
            self._std_weight_position * mean[3],   # u
            self._std_weight_position * mean[3],   # v
            1e-2,                                   # γ
            self._std_weight_position * mean[3],   # h
        ]
        std_vel = [
            self._std_weight_velocity * mean[3],   # u_dot
            self._std_weight_velocity * mean[3],   # v_dot
            1e-5,                                   # γ_dot
            self._std_weight_velocity * mean[3],   # h_dot
        ]
        motion_cov = np.diag(np.square(np.r_[std_pos, std_vel]))  # Q

        # Predict
        mean       = self._motion_mat @ mean                              # F @ x
        covariance = (self._motion_mat @ covariance @
                      self._motion_mat.T + motion_cov)                    # F@P@Fᵀ + Q

        return mean, covariance

    def project(self, mean, covariance):
        """
        Project state to measurement space.

        μ_z = H @ x
        S   = H @ P @ Hᵀ + R

        Returns: (projected_mean, innovation_covariance)
        """
        # Measurement noise R
        std = [
            self._std_weight_position * mean[3],   # u
            self._std_weight_position * mean[3],   # v
            1e-1,                                   # γ
            self._std_weight_position * mean[3],   # h
        ]
        innovation_cov = np.diag(np.square(std))   # R

        mean       = self._update_mat @ mean                              # H @ x
        covariance = (self._update_mat @ covariance @
                      self._update_mat.T + innovation_cov)                # H@P@Hᵀ + R

        return mean, covariance

    def update(self, mean, covariance, measurement):
        """
        Kalman UPDATE step.

        y = z - H @ x̂              (innovation)
        S = H @ P̂ @ Hᵀ + R         (innovation covariance)
        K = P̂ @ Hᵀ @ S⁻¹           (Kalman gain)
        x = x̂ + K @ y              (updated mean)
        P = (I - K @ H) @ P̂        (updated covariance)

        Returns: (updated_mean, updated_covariance)
        """
        projected_mean, projected_cov = self.project(mean, covariance)

        # Solve for Kalman gain: K = P @ Hᵀ @ S⁻¹
        # Equivalent to solving: Sᵀ @ Kᵀ = (P @ Hᵀ)ᵀ
        chol_factor, lower = scipy.linalg.cho_factor(
            projected_cov, lower=True, check_finite=False
        )
        kalman_gain = scipy.linalg.cho_solve(
            (chol_factor, lower),
            (covariance @ self._update_mat.T).T,
            check_finite=False
        ).T
        # kalman_gain = K  shape (8, 4)

        # Innovation
        innovation = measurement - projected_mean   # y = z - H@x̂

        # Update
        new_mean       = mean + innovation @ kalman_gain.T        # x = x̂ + K@y
        new_covariance = covariance - kalman_gain @ projected_cov @ kalman_gain.T

        return new_mean, new_covariance

    def gating_distance(self, mean, covariance, measurements,
                        only_position=False):
        """
        Compute Mahalanobis distance between state and measurements.
        Used to gate unlikely matches.

        Returns: array of distances, one per measurement
        """
        projected_mean, projected_cov = self.project(mean, covariance)

        if only_position:
            # Only use u,v (ignore γ, h)
            projected_mean = projected_mean[:2]
            projected_cov  = projected_cov[:2, :2]
            measurements   = measurements[:, :2]

        # Cholesky decomposition for efficient computation
        cholesky_factor = np.linalg.cholesky(projected_cov)

        # d = measurement - projected_mean
        d = measurements - projected_mean

        # Solve: L @ z = dᵀ  →  z = L⁻¹ @ dᵀ
        z = scipy.linalg.solve_triangular(
            cholesky_factor, d.T,
            lower=True, check_finite=False,
            overwrite_b=True
        )

        # Mahalanobis distance = sum(z²) per measurement
        squared_maha = np.sum(z * z, axis=0)
        return squared_maha

Writing deepsort_object_tracking/kalman_filter.py


In [4]:
%%writefile deepsort_object_tracking/nn_matching.py
import numpy as np


def cosine_distance(a, b, data_is_normalized=False):
    """
    Compute cosine distance between rows of a and b.

    a: (N, D) matrix
    b: (M, D) matrix

    Returns: (N, M) distance matrix
    """
    if not data_is_normalized:
        # L2 normalize each row
        a = a / np.linalg.norm(a, axis=1, keepdims=True)
        b = b / np.linalg.norm(b, axis=1, keepdims=True)

    # Cosine similarity = a @ bᵀ
    # Cosine distance   = 1 - similarity
    return 1.0 - np.dot(a, b.T)


def nn_cosine_distance(x, y):
    """
    Nearest-neighbor cosine distance.

    x: gallery features (N_gallery, D)
    y: query features   (N_query, D)

    For each query, find minimum distance to gallery.
    Returns: (N_gallery,) array of minimum distances
    """
    distances = cosine_distance(x, y)          # (N_gallery, N_query)
    return distances.min(axis=0)               # min over gallery → (N_query,)


class NearestNeighborDistanceMetric:
    """
    Maintains a feature gallery per track.
    Computes nearest-neighbor cosine distance for matching.
    """

    def __init__(self, matching_threshold, budget=None):
        """
        matching_threshold: max cosine distance for valid match
                            paper uses 0.2
        budget:             max features to store per track (None=unlimited)
                            paper uses 100
        """
        self.matching_threshold = matching_threshold
        self.budget             = budget
        self.samples            = {}    # track_id → list of features

    def partial_fit(self, features, targets, active_targets):
        """
        Update gallery with new features.

        features:       (N, 128) new embeddings
        targets:        (N,) track IDs for each feature
        active_targets: list of currently active track IDs
        """
        for feature, target in zip(features, targets):
            self.samples.setdefault(target, [])
            self.samples[target].append(feature)

            # Keep only last 'budget' features per track
            if self.budget is not None:
                self.samples[target] = self.samples[target][-self.budget:]

        # Remove features for inactive tracks (save memory)
        self.samples = {
            k: v for k, v in self.samples.items()
            if k in active_targets
        }

    def distance(self, features, targets):
        """
        Compute cost matrix between detections and tracks.

        features: (N, 128) detection embeddings
        targets:  list of track IDs

        Returns: (len(targets), N) cost matrix
        """
        cost_matrix = np.zeros((len(targets), len(features)))

        for i, target in enumerate(targets):
            gallery = np.array(self.samples[target])   # (K, 128)
            # Min cosine distance over gallery
            cost_matrix[i] = nn_cosine_distance(gallery, features)

        return cost_matrix

Writing deepsort_object_tracking/nn_matching.py


In [5]:
%%writefile deepsort_object_tracking/linear_assignment.py
import numpy as np
from scipy.optimize import linear_sum_assignment
from kalman_filter import chi2inv95

INFTY_COST = 1e5    # represents impossible assignment


def min_cost_matching(distance_metric, max_distance,
                      tracks, detections,
                      track_indices=None,
                      detection_indices=None):
    """
    Solve linear assignment problem.

    Returns: (matches, unmatched_tracks, unmatched_detections)
    """
    if track_indices is None:
        track_indices = list(range(len(tracks)))
    if detection_indices is None:
        detection_indices = list(range(len(detections)))

    if len(track_indices) == 0 or len(detection_indices) == 0:
        return [], track_indices, detection_indices

    # Build cost matrix
    cost_matrix = distance_metric(
        tracks, detections,
        track_indices, detection_indices
    )

    # Set costs above threshold to INFTY (impossible)
    cost_matrix[cost_matrix > max_distance] = max_distance + 1e-5

    # Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches            = []
    unmatched_tracks   = list(track_indices)
    unmatched_dets     = list(detection_indices)

    for r, c in zip(row_ind, col_ind):
        track_idx = track_indices[r]
        det_idx   = detection_indices[c]

        if cost_matrix[r, c] > max_distance:
            # Cost too high → unmatched
            continue

        matches.append((track_idx, det_idx))
        unmatched_tracks.remove(track_idx)
        unmatched_dets.remove(det_idx)

    return matches, unmatched_tracks, unmatched_dets


def matching_cascade(distance_metric, max_distance,
                     cascade_depth, tracks, detections,
                     track_indices=None, detection_indices=None):
    """
    Cascade matching — prioritize recently seen tracks.

    cascade_depth = max_age (70 in paper)

    For age=1,2,...,max_age:
        Try to match tracks of this age to remaining detections
    """
    if track_indices is None:
        track_indices = list(range(len(tracks)))
    if detection_indices is None:
        detection_indices = list(range(len(detections)))

    unmatched_detections = list(detection_indices)
    matches = []

    # Priority: tracks missed for fewer frames go first
    for level in range(cascade_depth):
        if len(unmatched_detections) == 0:
            break   # all detections matched

        # Tracks missed for exactly 'level+1' frames
        track_indices_l = [
            k for k in track_indices
            if tracks[k].time_since_update == level + 1
        ]

        if len(track_indices_l) == 0:
            continue   # no tracks at this age level

        # Match these tracks to remaining unmatched detections
        matches_l, _, unmatched_detections = min_cost_matching(
            distance_metric, max_distance,
            tracks, detections,
            track_indices_l, unmatched_detections
        )

        matches += matches_l

    # Remaining unmatched tracks
    unmatched_tracks = list(set(track_indices) - {k for k, _ in matches})

    return matches, unmatched_tracks, unmatched_detections


def gate_cost_matrix(kf, cost_matrix, tracks, detections,
                     track_indices, detection_indices,
                     gated_cost=INFTY_COST,
                     only_position=False):
    """
    Apply Mahalanobis gating to cost matrix.
    Set impossible matches (too far physically) to INFTY.
    """
    # Threshold from chi-squared distribution
    # 4 degrees of freedom (measurement dim)
    gating_threshold = chi2inv95[4]

    # Get all detection measurements
    measurements = np.array([
        detections[i].to_xyah()
        for i in detection_indices
    ])

    for row, track_idx in enumerate(track_indices):
        track = tracks[track_idx]

        # Compute Mahalanobis distances
        gating_distance = kf.gating_distance(
            track.mean, track.covariance,
            measurements, only_position
        )

        # Gate: set cost to INFTY if physically impossible
        cost_matrix[row, gating_distance > gating_threshold] = gated_cost

    return cost_matrix

Writing deepsort_object_tracking/linear_assignment.py


In [6]:
%%writefile deepsort_object_tracking/track.py
class TrackState:
    """Track lifecycle states."""
    Tentative = 1    # newly created, not confirmed
    Confirmed = 2    # matched n_init times, real object
    Deleted   = 3    # lost for max_age frames, remove


class Track:
    """
    Single tracked object.
    Maintained by the Tracker class.
    """

    def __init__(self, mean, covariance, track_id,
                 n_init, max_age, feature=None):
        """
        mean:        initial Kalman state (8,)
        covariance:  initial Kalman covariance (8,8)
        track_id:    unique integer ID
        n_init:      frames needed to confirm (3)
        max_age:     frames before deletion (70)
        feature:     initial appearance embedding (128,)
        """
        self.mean           = mean
        self.covariance     = covariance
        self.track_id       = track_id
        self.hits           = 1           # total matches
        self.age            = 1           # total frames
        self.time_since_update = 0        # frames since last match
        self.state          = TrackState.Tentative
        self.features       = []          # appearance gallery

        self._n_init  = n_init
        self._max_age = max_age

        if feature is not None:
            self.features.append(feature)

    def to_tlwh(self):
        """Get [top, left, width, height] from Kalman state."""
        ret    = self.mean[:4].copy()
        ret[2] *= ret[3]       # w = γ × h
        ret[:2] -= ret[2:] / 2 # top-left = center - size/2
        return ret

    def to_tlbr(self):
        """Get [x1, y1, x2, y2]."""
        ret = self.to_tlwh()
        ret[2:] = ret[:2] + ret[2:]
        return ret

    def predict(self, kf):
        """Run Kalman predict step."""
        self.mean, self.covariance = kf.predict(
            self.mean, self.covariance
        )
        self.age            += 1
        self.time_since_update += 1

    def update(self, kf, detection):
        """
        Run Kalman update step.
        Add detection feature to gallery.
        """
        self.mean, self.covariance = kf.update(
            self.mean, self.covariance,
            detection.to_xyah()
        )

        # Add to appearance gallery (keep last 100)
        self.features.append(detection.feature)
        if len(self.features) > 100:
            self.features.pop(0)

        self.hits              += 1
        self.time_since_update  = 0

        # State transitions
        if self.state == TrackState.Tentative and \
           self.hits >= self._n_init:
            self.state = TrackState.Confirmed

    def mark_missed(self):
        """Called when track has no matching detection."""
        if self.state == TrackState.Tentative:
            # Tentative tracks deleted immediately if missed
            self.state = TrackState.Deleted
        elif self.time_since_update > self._max_age:
            # Confirmed tracks deleted after max_age misses
            self.state = TrackState.Deleted

    def is_tentative(self):
        return self.state == TrackState.Tentative

    def is_confirmed(self):
        return self.state == TrackState.Confirmed

    def is_deleted(self):
        return self.state == TrackState.Deleted

Writing deepsort_object_tracking/track.py


In [7]:
%%writefile deepsort_object_tracking/tracker.py
import numpy as np
from kalman_filter import KalmanFilter
from track import Track, TrackState
from linear_assignment import (matching_cascade,
                                min_cost_matching,
                                gate_cost_matrix)
from nn_matching import NearestNeighborDistanceMetric


class Tracker:
    """
    Main DeepSORT tracker.
    Manages all Track objects.
    """

    def __init__(self, metric, max_iou_distance=0.7,
                 max_age=70, n_init=3):
        """
        metric:           NearestNeighborDistanceMetric instance
        max_iou_distance: IoU threshold for second stage matching
        max_age:          delete after this many missed frames
        n_init:           matches needed to confirm track
        """
        self.metric           = metric
        self.max_iou_distance = max_iou_distance
        self.max_age          = max_age
        self.n_init           = n_init

        self.kf      = KalmanFilter()
        self.tracks  = []
        self._next_id = 1

    def predict(self):
        """Propagate all track states one step forward."""
        for track in self.tracks:
            track.predict(self.kf)

    def update(self, detections):
        """
        Main update. Called every frame.

        detections: list of Detection objects
        """
        # ── Step 1: Cascade Matching (appearance) ────────
        matches_a, unmatched_tracks_a, unmatched_dets_a = \
            self._match(detections)

        # ── Step 2: IoU Matching for remaining tracks ─────
        # Unconfirmed tracks + confirmed tracks missed once
        iou_track_candidates = [
            i for i in unmatched_tracks_a
            if self.tracks[i].time_since_update == 1
        ]
        unmatched_tracks_b = [
            i for i in unmatched_tracks_a
            if self.tracks[i].time_since_update != 1
        ]

        matches_b, unmatched_tracks_b2, unmatched_dets_b = \
            min_cost_matching(
                self._iou_cost,
                self.max_iou_distance,
                self.tracks, detections,
                iou_track_candidates, unmatched_dets_a
            )

        matches = matches_a + matches_b
        unmatched_tracks = list(set(unmatched_tracks_b +
                                    unmatched_tracks_b2))

        # ── Step 3: Update matched tracks ─────────────────
        for track_idx, det_idx in matches:
            self.tracks[track_idx].update(
                self.kf, detections[det_idx]
            )

        # ── Step 4: Mark missed tracks ────────────────────
        for track_idx in unmatched_tracks:
            self.tracks[track_idx].mark_missed()

        # ── Step 5: Create new tracks ─────────────────────
        for det_idx in unmatched_dets_b:
            self._initiate_track(detections[det_idx])

        # ── Step 6: Delete dead tracks ────────────────────
        self.tracks = [t for t in self.tracks
                       if not t.is_deleted()]

        # ── Step 7: Update appearance gallery ────────────
        active_targets = [t.track_id for t in self.tracks
                          if t.is_confirmed()]
        features       = []
        targets        = []

        for track in self.tracks:
            if not track.is_confirmed():
                continue
            features += track.features
            targets  += [track.track_id] * len(track.features)
            track.features = []   # clear after storing

        self.metric.partial_fit(
            np.array(features),
            np.array(targets),
            active_targets
        )

    def _match(self, detections):
        """
        Cascade matching using appearance features + Mahalanobis gating.
        """
        def gated_metric(tracks, dets, track_indices, det_indices):
            # Get detection features
            features = np.array([dets[i].feature for i in det_indices])
            # Get track IDs for gallery lookup
            targets  = np.array([tracks[i].track_id for i in track_indices])

            # Appearance cost matrix (cosine distance)
            cost_matrix = self.metric.distance(features, targets)

            # Apply Mahalanobis gate
            cost_matrix = gate_cost_matrix(
                self.kf, cost_matrix, tracks, dets,
                track_indices, det_indices
            )
            return cost_matrix

        # Only confirmed tracks go into cascade matching
        confirmed_tracks = [
            i for i, t in enumerate(self.tracks)
            if t.is_confirmed()
        ]
        # Unconfirmed tracks go straight to IoU matching
        unconfirmed_tracks = [
            i for i, t in enumerate(self.tracks)
            if not t.is_confirmed()
        ]

        # Cascade matching for confirmed tracks
        matches_a, unmatched_tracks_a, unmatched_dets_a = \
            matching_cascade(
                gated_metric,
                self.metric.matching_threshold,
                self.max_age,
                self.tracks, detections,
                confirmed_tracks
            )

        # IoU matching for unconfirmed + recently lost confirmed
        iou_candidates = unconfirmed_tracks + [
            k for k in unmatched_tracks_a
            if self.tracks[k].time_since_update == 1
        ]

        matches_b, unmatched_tracks_b, unmatched_dets_b = \
            min_cost_matching(
                self._iou_cost,
                self.max_iou_distance,
                self.tracks, detections,
                iou_candidates, unmatched_dets_a
            )

        matches = matches_a + matches_b
        unmatched_tracks = list(
            set(unmatched_tracks_a) - {k for k, _ in matches_b}
        ) + unmatched_tracks_b

        return matches, unmatched_tracks, unmatched_dets_b

    def _iou_cost(self, tracks, detections,
                  track_indices, detection_indices):
        """
        IoU-based cost matrix for second stage matching.
        1 - IoU so that lower = better match.
        """
        cost_matrix = np.zeros((len(track_indices),
                                len(detection_indices)))

        for row, t_idx in enumerate(track_indices):
            if tracks[t_idx].time_since_update > 1:
                cost_matrix[row, :] = 1e5
                continue

            bbox = tracks[t_idx].to_tlbr()

            for col, d_idx in enumerate(detection_indices):
                det_bbox = detections[d_idx].bbox
                cost_matrix[row, col] = 1 - self._iou(bbox, det_bbox)

        return cost_matrix

    def _iou(self, bbox1, bbox2):
        """IoU between two [x1,y1,x2,y2] boxes."""
        xx1 = max(bbox1[0], bbox2[0])
        yy1 = max(bbox1[1], bbox2[1])
        xx2 = min(bbox1[2], bbox2[2])
        yy2 = min(bbox1[3], bbox2[3])

        w = max(0., xx2 - xx1)
        h = max(0., yy2 - yy1)
        intersection = w * h

        area1 = ((bbox1[2]-bbox1[0]) * (bbox1[3]-bbox1[1]))
        area2 = ((bbox2[2]-bbox2[0]) * (bbox2[3]-bbox2[1]))
        union = area1 + area2 - intersection

        return intersection / (union + 1e-6)

    def _initiate_track(self, detection):
        """Create a new track from an unmatched detection."""
        mean, covariance = self.kf.initiate(detection.to_xyah())
        self.tracks.append(Track(
            mean, covariance,
            self._next_id,
            self.n_init,
            self.max_age,
            detection.feature
        ))
        self._next_id += 1

Writing deepsort_object_tracking/tracker.py


In [8]:
%%writefile deepsort_object_tracking/extractor.py
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
import cv2


class ResNetBlock(nn.Module):
    """Small residual block for Re-ID CNN."""

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))   # skip connection!


class ReIDNet(nn.Module):
    """
    Lightweight Re-ID CNN.
    Input:  (B, 3, 128, 64) — person crop
    Output: (B, 128)        — appearance embedding
    """

    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),                # 64×32

            # Block 2
            nn.Conv2d(32, 64, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            ResNetBlock(64),
            nn.MaxPool2d(2, 2),                # 32×16

            # Block 3
            nn.Conv2d(64, 128, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            ResNetBlock(128),
            nn.MaxPool2d(2, 2),                # 16×8

            # Block 4
            nn.Conv2d(128, 128, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            ResNetBlock(128),
        )
        self.pool    = nn.AdaptiveAvgPool2d((1, 1))
        self.embed   = nn.Linear(128, 128)

    def forward(self, x):
        x = self.backbone(x)       # (B, 128, 16, 8)
        x = self.pool(x)           # (B, 128, 1, 1)
        x = x.flatten(1)           # (B, 128)
        x = self.embed(x)          # (B, 128)
        # L2 normalize → unit vectors for cosine distance
        x = nn.functional.normalize(x, p=2, dim=1)
        return x


class FeatureExtractor:
    """
    Extracts appearance embeddings from detection crops.
    """

    def __init__(self, model_path=None, device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        self.model = ReIDNet().to(device)
        self.model.eval()

        if model_path:
            self.model.load_state_dict(
                torch.load(model_path, map_location=device)
            )

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((128, 64)),   # height×width for person
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet stats
                std=[0.229, 0.224, 0.225]
            )
        ])

    @torch.no_grad()
    def __call__(self, frame, bboxes):
        """
        Extract features for all detections in one frame.

        frame:  BGR image (H, W, 3)
        bboxes: list of [x1, y1, x2, y2]

        Returns: (N, 128) numpy array
        """
        if len(bboxes) == 0:
            return np.array([])

        crops = []
        for bbox in bboxes:
            x1, y1, x2, y2 = [int(v) for v in bbox]

            # Clamp to frame boundaries
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(frame.shape[1], x2)
            y2 = min(frame.shape[0], y2)

            crop = frame[y1:y2, x1:x2]   # BGR crop

            if crop.size == 0:
                crop = np.zeros((128, 64, 3), dtype=np.uint8)

            # BGR → RGB
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            crops.append(self.transform(crop))

        # Stack into batch
        batch = torch.stack(crops).to(self.device)   # (N, 3, 128, 64)

        # Extract embeddings
        embeddings = self.model(batch)                # (N, 128)
        return embeddings.cpu().numpy()

Writing deepsort_object_tracking/extractor.py


In [9]:
%%writefile deepsort_object_tracking/main.py
import cv2
import numpy as np
from deep_sort.tracker import Tracker
from deep_sort.detection import Detection
from deep_sort.nn_matching import NearestNeighborDistanceMetric
from appearance.extractor import FeatureExtractor
from detector import YOLOv3Detector


np.random.seed(42)
COLORS = np.random.randint(0, 255, size=(300, 3))


def draw_tracks(frame, tracks):
    for track in tracks:
        if not track.is_confirmed():
            continue

        bbox     = track.to_tlbr()
        track_id = track.track_id
        x1,y1,x2,y2 = [int(v) for v in bbox]

        color = COLORS[track_id % len(COLORS)].tolist()
        cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
        cv2.putText(
            frame, f"ID:{track_id}",
            (x1, y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6, color, 2
        )
    return frame


def run_tracking(video_path, output_path=None, model_path=None):

    # ── Initialize components ─────────────────────────────
    detector  = YOLOv3Detector(conf_threshold=0.5,
                                classes=['person','car','truck'])

    extractor = FeatureExtractor(model_path=model_path)

    metric    = NearestNeighborDistanceMetric(
        matching_threshold=0.2,   # cosine distance threshold
        budget=100                # max features per track gallery
    )

    tracker   = Tracker(
        metric=metric,
        max_iou_distance=0.7,
        max_age=70,               # from paper
        n_init=3                  # from paper
    )

    cap = cv2.VideoCapture(video_path)

    if output_path:
        fps    = int(cap.get(cv2.CAP_PROP_FPS))
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        writer = cv2.VideoWriter(
            output_path,
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps, (width, height)
        )

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        print(f"Frame {frame_idx}")

        # ── Step 1: Detect ────────────────────────────────
        raw_dets = detector.detect(frame)
        # raw_dets: (N, 5) [x1,y1,x2,y2,conf]

        # ── Step 2: Extract appearance features ──────────
        if len(raw_dets) > 0:
            bboxes   = raw_dets[:, :4]
            features = extractor(frame, bboxes)   # (N, 128)
        else:
            features = []

        # ── Step 3: Build Detection objects ───────────────
        detections = []
        for i, det in enumerate(raw_dets):
            x1,y1,x2,y2,conf = det
            detections.append(Detection(
                bbox=[x1,y1,x2,y2],
                confidence=conf,
                feature=features[i]
            ))

        # ── Step 4: Kalman predict ────────────────────────
        tracker.predict()

        # ── Step 5: Update tracker ────────────────────────
        tracker.update(detections)

        # ── Step 6: Visualize ─────────────────────────────
        frame = draw_tracks(frame, tracker.tracks)

        confirmed = sum(1 for t in tracker.tracks if t.is_confirmed())
        cv2.putText(
            frame,
            f"Detections:{len(detections)}  Tracks:{confirmed}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1, (255,255,255), 2
        )

        cv2.imshow("DeepSORT", frame)
        if output_path:
            writer.write(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    if output_path:
        writer.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    for video_path in input_folder.glob("*.mp4"):
        output_path = output_folder / video_path.name
    
        run_tracking(
            video_path=str(video_path),
            output_path=str(output_path),
            model_path=None
        )

Writing deepsort_object_tracking/main.py


In [10]:
"""
Summary of what is going on
Frame arrives
      ↓
YOLOv3 → (N,5) [x1,y1,x2,y2,conf]
      ↓
Re-ID CNN → (N,128) appearance embeddings
      ↓
Build Detection objects [(bbox, conf, feature) × N]
      ↓
tracker.predict()
  → every Track runs Kalman PREDICT step
  → Kalman covariance grows (more uncertain)
      ↓
tracker.update(detections):
      ↓
  STAGE 1 — Cascade Matching (confirmed tracks only)
    for age = 1, 2, ..., 70:
      tracks missed for 'age' frames
        ↓
      cost = cosine_distance(det_features, track_gallery)
        ↓
      gate with Mahalanobis distance (χ²=9.4877)
        ↓
      Hungarian algorithm → optimal assignment
        ↓
      matched detections removed from pool
      ↓
  STAGE 2 — IoU Matching (unconfirmed + recently lost)
    remaining detections vs unconfirmed tracks
        ↓
      cost = 1 - IoU
        ↓
      Hungarian algorithm
      ↓
  Update matched tracks:
    Kalman UPDATE step
    add feature to gallery
    increment hits
    TENTATIVE → CONFIRMED if hits >= 3
      ↓
  Mark missed tracks:
    TENTATIVE → DELETED immediately
    CONFIRMED → DELETED if time_since_update > 70
      ↓
  Create new tracks:
    unmatched detections → new TENTATIVE tracks
      ↓
  Output: all CONFIRMED tracks
      ↓
Draw boxes + IDs
"""

"\nSummary of what is going on\nFrame arrives\n      ↓\nYOLOv3 → (N,5) [x1,y1,x2,y2,conf]\n      ↓\nRe-ID CNN → (N,128) appearance embeddings\n      ↓\nBuild Detection objects [(bbox, conf, feature) × N]\n      ↓\ntracker.predict()\n  → every Track runs Kalman PREDICT step\n  → Kalman covariance grows (more uncertain)\n      ↓\ntracker.update(detections):\n      ↓\n  STAGE 1 — Cascade Matching (confirmed tracks only)\n    for age = 1, 2, ..., 70:\n      tracks missed for 'age' frames\n        ↓\n      cost = cosine_distance(det_features, track_gallery)\n        ↓\n      gate with Mahalanobis distance (χ²=9.4877)\n        ↓\n      Hungarian algorithm → optimal assignment\n        ↓\n      matched detections removed from pool\n      ↓\n  STAGE 2 — IoU Matching (unconfirmed + recently lost)\n    remaining detections vs unconfirmed tracks\n        ↓\n      cost = 1 - IoU\n        ↓\n      Hungarian algorithm\n      ↓\n  Update matched tracks:\n    Kalman UPDATE step\n    add feature to gal

In [11]:
!python deepsort_object_tracking/main.py

Traceback (most recent call last):
  File "c:\Users\ABDALAH\Desktop\projects\deepsort_object_tracking\main.py", line 1, in <module>
    import cv2
ModuleNotFoundError: No module named 'cv2'
